In [9]:
using Pkg
Pkg.activate(".")

using Flux          # Neural network library (like PyTorch for Julia)
using NonlinearSolve # SciML's nonlinear solver suite
using Zygote        # Automatic differentiation (Flux's backend)
using Optimisers    # Optimizer algorithms (Adam, SGD, etc.)
using ImplicitDifferentiation	# Library for IFT powered backward
using Random        # For reproducibility
using LinearAlgebra # For norm()
using Infiltrator
using Debugger

Random.seed!(42)

# ============================================================
# PART 1: THE NONLINEAR PROBLEM
# ============================================================

function phi(x, y)
    r1 = y[1]^3 + x[1] * y[2] - 1.0
    r2 = y[2]^3 + x[2] * y[1] - 1.0
    return [r1, r2]
end

# ============================================================
# PART 2: WRAP THE SOLVER FOR IFT
# ============================================================
#
# ImplicitDifferentiation.jl needs two things:
#
#   1. A "forward" function: given input p, produce output y
#      (this runs the solver)
#
#   2. A "conditions" function: F(p, y) = 0 defines the implicit equation
#      (this is just phi — the residual that equals zero at the solution)
#
# Then IFT gives us: dy/dp = -(∂F/∂y)⁻¹ (∂F/∂p)
# Zygote never touches the solver internals.

"""
Forward map: takes a parameter vector p and returns the solver solution.
p will contain BOTH x and y0 (the warm start) packed together.
"""
function forward_solver(p)
	x = p[1:2]
	y0 = p[3:4]

    residual!(u, params) = phi(params, u)
    prob = NonlinearProblem(residual!, Float64.(y0), Float64.(x))
    sol = solve(prob, NewtonRaphson(); abstol=1e-10)
    return (sol.u, nothing)
end

function forward_solver_with_iters(p)
	x = p[1:2]
	y0 = p[3:4]

    residual!(u, params) = phi(params, u)
    prob = NonlinearProblem(residual!, Float64.(y0), Float64.(x))
    sol = solve(prob, NewtonRaphson(); abstol=1e-10)

	# Calculate number of iters
	n_iters = sol.stats.nsteps

    return (sol.u, n_iters)
end

"""
Conditions: the implicit equation F(p, y) = 0 that defines the solution.
At convergence, phi(x, y*) = 0.
"""
function conditions(p, y, z)
    x = p[1:2]
    return phi(x, y)
end

# Create the implicit function: this is now a DIFFERENTIABLE solver
# ImplicitDifferentiation.jl handles the IFT automatically
implicit_solver = ImplicitFunction(forward_solver, conditions)

# ============================================================
# PART 3: NEURAL NETWORK
# ============================================================

n_input = 2
n_output = 2

model = Chain(
    Dense(n_input, 64, relu),
    Dense(64, 64, relu),
    Dense(64, n_output)
)

# ============================================================
# PART 4: LOSS FUNCTION — SOLVER FULLY IN THE LOOP
# ============================================================
#
# L(θ) = ||phi(x, nn(x;θ))||² + λ * ||nn(x;θ) - solver(x; y0=nn(x;θ))||²
#
# Gradient flows through BOTH terms, including through the solver
# output via IFT. The computation graph looks like:
#
#                   ┌──────── phi(x, y_hat) ──► physics_loss
#                   │
#   x ──► nn(x) ──► y_hat ──────────────────────────► MSE ──► loss
#            │                                          ▲
#            │      gradient via IFT                    │
#            └──► implicit_solver([x; y_hat]) ──► y_star
#                 (differentiable!)

λ = 0.1

function loss_single(m, x)
    y_hat = m(x)

    # Term 1: Physics residual
    residual = phi(x, y_hat)
    physics_loss = sum(residual .^ 2)

    # Term 2: MSE with solver in the loop
    # This call IS differentiable — IFT computes dy_star/dp for us
	p = vcat(x, y_hat)
    (y_star, _) = implicit_solver(p)

    supervised_loss = sum((y_hat .- y_star) .^ 2)

    return physics_loss + λ * supervised_loss
end

# ============================================================
# PART 5: TRAINING
# ============================================================

n_samples = 200
X_data = [rand(n_input) .- 0.5 for _ in 1:n_samples]

# Generate a separate validation set (unseen during training)
n_val = 50
X_val = [rand(n_input) .- 0.5 for _ in 1:n_val]

opt = Adam(1e-3)
opt_state = Optimisers.setup(opt, model)

function train!(model, opt_state, X_data; n_epochs=500, batch_size=32)
	n_samples = length(X_data)

    for epoch in 1:n_epochs
        perm = randperm(n_samples)
        epoch_loss = 0.0
        n_batches = 0

        for batch_start in 1:batch_size:n_samples
            batch_end = min(batch_start + batch_size - 1, n_samples)
            X_batch = X_data[perm[batch_start:batch_end]]

            loss_val, grads = Zygote.withgradient(model) do m
                total = 0.0
                for x in X_batch
                    total += loss_single(m, x)
                end
                total / length(X_batch)
            end

            opt_state, model = Optimisers.update(opt_state, model, grads[1])

            epoch_loss += loss_val
            n_batches += 1
        end

        if epoch % 50 == 1 || epoch == n_epochs
            avg_loss = epoch_loss / n_batches
            println("Epoch $(lpad(epoch, 4))/$(n_epochs) | Loss: $(round(avg_loss; digits=6))")
        end
    end

    return model, opt_state
end

n_epochs=500
batch_size=32

model, opt_state = train!(
	model, opt_state, X_data;
	n_epochs=n_epochs, batch_size=batch_size
)

# ============================================================
# PART 6: EVALUATION
# ============================================================

println("\n" * "="^50)
println("EVALUATION")
println("="^50)

total_residual_norm = 0.0
total_mse = 0.0
total_loss = 0.0
cold_iters_total = 0
nn_iters_total = 0
cold_start = [1.0, 1.0]

for i in 1:5
    x = X_data[i]
    y_hat = model(x)
    y_star = forward_solver(vcat(x, y_hat))[1]

    println("\nSample $i:")
    println("  y_hat  (nn)     = $(round.(y_hat; digits=6))")
    println("  y_star (solver) = $(round.(y_star; digits=6))")
    println("  ||y_hat - y_star|| = $(round(norm(y_hat .- y_star); digits=8))")
    println("  phi(x, y_hat)      = $(round.(phi(x, y_hat); digits=8))")
end

for i in 1:n_val
    x = X_val[i]
    y_hat = model(x)

    # Solve with NN warm start
    _, nn_iters = forward_solver_with_iters(vcat(Float64.(x), Float64.(y_hat)))

    # Solve with cold start
    _, cold_iters = forward_solver_with_iters(vcat(Float64.(x), cold_start))

    # Solver solution using nn prediction as warm start
    y_star, _ = forward_solver(vcat(Float64.(x), Float64.(y_hat)))

    # Metrics
    residual = phi(x, y_hat)
    residual_norm = sum(residual .^ 2)
    mse = sum((y_hat .- y_star) .^ 2)
    loss = residual_norm + λ * mse

    total_residual_norm += residual_norm
    total_mse += mse
    total_loss += loss
    nn_iters_total += nn_iters
    cold_iters_total += cold_iters

    # Print details for first 5 samples
    if i <= 5
        println("\nSample $i:")
        println("  x              = $(round.(x; digits=4))")
        println("  y_hat  (nn)    = $(round.(y_hat; digits=6))")
        println("  y_star (solver)= $(round.(y_star; digits=6))")
        println("  ||phi(x, y_hat)||² = $(round(residual_norm; digits=8))")
        println("  MSE(y_hat, y*)     = $(round(mse; digits=8))")
        println("  Loss               = $(round(loss; digits=8))")
        println("  Iterations: cold=$(cold_iters), nn=$(nn_iters)")
    end
end

# Aggregate metrics
println("\n" * "-"^60)
println("AGGREGATE METRICS (averaged over $(n_val) samples)")
println("-"^60)
println("  Avg ||phi(x, y_hat)||² (residual) = $(round(total_residual_norm / n_val; digits=8))")
println("  Avg MSE(y_hat, y*)                = $(round(total_mse / n_val; digits=8))")
println("  Avg Loss                          = $(round(total_loss / n_val; digits=8))")

println("\n" * "-"^60)
println("ITERATION COUNT: NN warm start vs cold start $(cold_start)")
println("-"^60)
println("  Avg iterations (cold start) = $(round(cold_iters_total / n_val; digits=2))")
println("  Avg iterations (NN start)   = $(round(nn_iters_total / n_val; digits=2))")
println("  Iteration savings           = $(round((1 - nn_iters_total/cold_iters_total) * 100; digits=1))%")

  Activating project at `~/Desktop/Projects/18.337/18337-final-project`
┌ Warning: Layer with Float32 parameters got Float64 input.
│   The input will be converted, but any earlier layers may be very slow.
│   layer = Dense(2 => 64, relu)
│   summary(x) = 2-element Vector{Float64}
└ @ Flux /home/vaishnavi/.julia/packages/Flux/hrg9M/src/layers/stateless.jl:60


Epoch    1/500 | Loss: 2.157577
Epoch   51/500 | Loss: 0.001004
Epoch  101/500 | Loss: 0.000267
Epoch  151/500 | Loss: 0.000159
Epoch  201/500 | Loss: 8.7e-5
Epoch  251/500 | Loss: 5.7e-5
Epoch  301/500 | Loss: 8.9e-5
Epoch  351/500 | Loss: 9.4e-5
Epoch  401/500 | Loss: 4.5e-5
Epoch  451/500 | Loss: 2.7e-5
Epoch  500/500 | Loss: 2.5e-5

EVALUATION

Sample 1:
  y_hat  (nn)     = Float32[1.128232, 0.876538]
  y_star (solver) = [1.127538, 0.8747]
  ||y_hat - y_star|| = 0.00196388
  phi(x, y_hat)      = [0.00173607, 0.00442965]

Sample 2:
  y_hat  (nn)     = Float32[0.802733, 1.058389]
  y_star (solver) = [0.806321, 1.058469]
  ||y_hat - y_star|| = 0.00358935
  phi(x, y_hat)      = [-0.00700398, 0.00055857]

Sample 3:
  y_hat  (nn)     = Float32[1.063372, 0.84235]
  y_star (solver) = [1.064464, 0.841983]
  ||y_hat - y_star|| = 0.001152
  phi(x, y_hat)      = [-0.00379804, 0.00036693]

Sample 4:
  y_hat  (nn)     = Float32[1.052807, 0.955388]
  y_star (solver) = [1.054597, 0.954119]
  ||y_h

## 5x5 Case

In [11]:
using Pkg
Pkg.activate(".")

using Flux
using NonlinearSolve
using Zygote
using Optimisers
using ImplicitDifferentiation
using Random
using LinearAlgebra

Random.seed!(42)

# ============================================================
# PART 1: THE NONLINEAR PROBLEM (5D)
# ============================================================
#
# A coupled system with circular structure:
# Each equation has a cubic self-term and two cross-coupling
# terms mediated by the input parameters x.
#
# This is a natural extension of the 2D system:
#   2D: y[i]^3 + x[i]*y[j] - 1 = 0
#   5D: y[i]^3 + x[i]*y[i+1] + 0.5*x[i+1]*y[i+2] - 1 = 0
#        (indices wrap around cyclically)

n_dim = 5  # Problem dimension

function phi(x, y)
    return [
        y[i]^3 + x[i] * y[mod1(i + 1, n_dim)] + 0.5 * x[mod1(i + 1, n_dim)] * y[mod1(i + 2, n_dim)] - 1.0
        for i in 1:n_dim
    ]
end


# ============================================================
# PART 2: WRAP THE SOLVER FOR IFT
# ============================================================

function forward_solver(p)
    x = p[1:n_dim]
    y0 = p[n_dim+1:2*n_dim]

    residual!(u, params) = phi(params, u)
    prob = NonlinearProblem(residual!, Float64.(y0), Float64.(x))
    sol = solve(prob, NewtonRaphson(); abstol=1e-10)
    return (sol.u, nothing)
end

function forward_solver_with_iters(p)
    x = p[1:n_dim]
    y0 = p[n_dim+1:2*n_dim]

    residual!(u, params) = phi(params, u)
    prob = NonlinearProblem(residual!, Float64.(y0), Float64.(x))
    sol = solve(prob, NewtonRaphson(); abstol=1e-10)

    n_iters = sol.stats.nsteps
    return (sol.u, n_iters)
end

function conditions(p, y, z)
    x = p[1:n_dim]
    return phi(x, y)
end

implicit_solver = ImplicitFunction(forward_solver, conditions)

# ============================================================
# PART 3: NEURAL NETWORK (scaled up)
# ============================================================

n_input = n_dim
n_output = n_dim

model = Chain(
    Dense(n_input, 128, relu),
    Dense(128, 128, relu),
    Dense(128, 64, relu),
    Dense(64, n_output)
)

# ============================================================
# PART 4: LOSS FUNCTION
# ============================================================

λ = 0.1

function loss_single(m, x)
    y_hat = m(x)

    # Term 1: Physics residual
    residual = phi(x, y_hat)
    physics_loss = sum(residual .^ 2)

    # Term 2: Fixed-point loss (solver in the loop via IFT)
    p = vcat(x, y_hat)
    (y_star, _) = implicit_solver(p)
    supervised_loss = sum((y_hat .- y_star) .^ 2)

    return physics_loss + λ * supervised_loss
end

# ============================================================
# PART 5: TRAINING
# ============================================================

# More samples since the problem is harder
n_samples = 500
X_data = [rand(Float32, n_input) .- 0.5f0 for _ in 1:n_samples]

n_val = 100
X_val = [rand(Float32, n_input) .- 0.5f0 for _ in 1:n_val]

opt = Adam(1e-3)
opt_state = Optimisers.setup(opt, model)

function train!(model, opt_state, X_data; n_epochs=800, batch_size=32)
    n_samples = length(X_data)

    for epoch in 1:n_epochs
        perm = randperm(n_samples)
        epoch_loss = 0.0
        n_batches = 0

        for batch_start in 1:batch_size:n_samples
            batch_end = min(batch_start + batch_size - 1, n_samples)
            X_batch = X_data[perm[batch_start:batch_end]]

            loss_val, grads = Zygote.withgradient(model) do m
                total = 0.0
                for x in X_batch
                    total += loss_single(m, x)
                end
                total / length(X_batch)
            end

            opt_state, model = Optimisers.update(opt_state, model, grads[1])

            epoch_loss += loss_val
            n_batches += 1
        end

        if epoch % 50 == 1 || epoch == n_epochs
            avg_loss = epoch_loss / n_batches
            println("Epoch $(lpad(epoch, 4))/$(n_epochs) | Loss: $(round(avg_loss; digits=6))")
        end
    end

    return model, opt_state
end

n_epochs = 800
batch_size = 32

println("Training on $(n_dim)D nonlinear system...")
println("Network: $(n_input) → 128 → 128 → 64 → $(n_output)")
println("Training samples: $(n_samples), Validation: $(n_val)")
println()

model, opt_state = train!(
    model, opt_state, X_data;
    n_epochs=n_epochs, batch_size=batch_size
)

# ============================================================
# PART 6: EVALUATION
# ============================================================

println("\n" * "="^60)
println("EVALUATION ($(n_dim)D System)")
println("="^60)

total_residual_norm = 0.0
total_mse = 0.0
total_loss = 0.0
cold_iters_total = 0
nn_iters_total = 0
cold_start = ones(Float64, n_dim)  # [1.0, 1.0, 1.0, 1.0, 1.0]

for i in 1:n_val
    x = X_val[i]
    y_hat = model(x)

    # Solve with NN warm start
    _, nn_iters = forward_solver_with_iters(vcat(Float64.(x), Float64.(y_hat)))

    # Solve with cold start
    _, cold_iters = forward_solver_with_iters(vcat(Float64.(x), cold_start))

    # Solver solution using nn prediction as warm start
    y_star, _ = forward_solver(vcat(Float64.(x), Float64.(y_hat)))

    # Metrics
    residual = phi(x, y_hat)
    residual_norm = sum(residual .^ 2)
    mse = sum((y_hat .- y_star) .^ 2)
    loss = residual_norm + λ * mse

    total_residual_norm += residual_norm
    total_mse += mse
    total_loss += loss
    nn_iters_total += nn_iters
    cold_iters_total += cold_iters

    # Print details for first 5 samples
    if i <= 5
        println("\nSample $i:")
        println("  x              = $(round.(x; digits=4))")
        println("  y_hat  (nn)    = $(round.(y_hat; digits=6))")
        println("  y_star (solver)= $(round.(y_star; digits=6))")
        println("  ||phi(x, y_hat)||² = $(round(residual_norm; digits=8))")
        println("  MSE(y_hat, y*)     = $(round(mse; digits=8))")
        println("  Loss               = $(round(loss; digits=8))")
        println("  Iterations: cold=$(cold_iters), nn=$(nn_iters)")
    end
end

# Aggregate metrics
println("\n" * "-"^60)
println("AGGREGATE METRICS (averaged over $(n_val) samples)")
println("-"^60)
println("  Avg ||phi(x, y_hat)||² (residual) = $(round(total_residual_norm / n_val; digits=8))")
println("  Avg MSE(y_hat, y*)                = $(round(total_mse / n_val; digits=8))")
println("  Avg Loss                          = $(round(total_loss / n_val; digits=8))")

println("\n" * "-"^60)
println("ITERATION COUNT: NN warm start vs cold start")
println("-"^60)
println("  Avg iterations (cold start) = $(round(cold_iters_total / n_val; digits=2))")
println("  Avg iterations (NN start)   = $(round(nn_iters_total / n_val; digits=2))")
println("  Iteration savings           = $(round((1 - nn_iters_total/cold_iters_total) * 100; digits=1))%")

  Activating project at `~/Desktop/Projects/18.337/18337-final-project`


Training on 5D nonlinear system...
Network: 5 → 128 → 128 → 64 → 5
Training samples: 500, Validation: 100

Epoch    1/800 | Loss: 5.094128
Epoch   51/800 | Loss: 0.002263
Epoch  101/800 | Loss: 0.000866
Epoch  151/800 | Loss: 0.001044
Epoch  201/800 | Loss: 0.000401
Epoch  251/800 | Loss: 0.000286
Epoch  301/800 | Loss: 0.000227
Epoch  351/800 | Loss: 0.000252
Epoch  401/800 | Loss: 0.000159
Epoch  451/800 | Loss: 0.000236
Epoch  501/800 | Loss: 0.000241
Epoch  551/800 | Loss: 0.000154
Epoch  601/800 | Loss: 0.000904
Epoch  651/800 | Loss: 0.000738
Epoch  701/800 | Loss: 0.000348
Epoch  751/800 | Loss: 0.000853
Epoch  800/800 | Loss: 0.000209

EVALUATION (5D System)

Sample 1:
  x              = Float32[0.0277, 0.3996, -0.034, -0.4103, -0.4853]
  y_hat  (nn)    = Float32[0.909871, 0.835434, 1.08346, 1.190097, 1.123404]
  y_star (solver)= [0.912709, 0.83742, 1.083389, 1.189698, 1.126964]
  ||phi(x, y_hat)||² = 0.00023159
  MSE(y_hat, y*)     = 2.484e-5
  Loss               = 0.00023408
